# Exploratory Data Analysis (EDA)

This notebook performs EDA for the Expedia hotel ranking competition using the processed parquet files:

- `../data/processed/training_set_VU_DM.parquet`
- `../data/processed/test_set_VU_DM.parquet`

## 0. Setup


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.5f}".format)

RANDOM_STATE = 42

DATA_DIR = Path("../data")
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find data/processed parquet files from the current working directory."
    )

TRAIN_PARQUET = DATA_DIR / "processed/training_set_VU_DM.parquet"
TEST_PARQUET = DATA_DIR / "processed/test_set_VU_DM.parquet"

print(f"Using data directory: {DATA_DIR.resolve()}")


Using data directory: /Users/mericmertbulca/dmt-2026-2nd-assignment/data


In [3]:
train_df = pd.read_parquet(TRAIN_PARQUET)
test_df = pd.read_parquet(TEST_PARQUET)

print(f"train shape: {train_df.shape[0]:,} rows x {train_df.shape[1]:,} columns")
print(f"test shape: {test_df.shape[0]:,} rows x {test_df.shape[1]:,} columns")
print(f"train memory: {(train_df.memory_usage(deep=True).sum() / 1024**2):,.2f} MB")
print(f"test memory: {(test_df.memory_usage(deep=True).sum() / 1024**2):,.2f} MB")


train shape: 4,958,347 rows x 54 columns
test shape: 4,959,183 rows x 50 columns
train memory: 690.38 MB
test memory: 652.66 MB


## 1. Schema and Train-only Columns

First check train/test column differences and mark train-only columns. Anything absent from test cannot be used as a final inference feature.


In [4]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
common_cols = sorted(train_cols & test_cols)
train_only_cols = sorted(train_cols - test_cols)
test_only_cols = sorted(test_cols - train_cols)

schema_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(train_df), len(test_df)],
        "columns": [train_df.shape[1], test_df.shape[1]],
        "unique_srch_id": [train_df["srch_id"].nunique(), test_df["srch_id"].nunique()],
        "unique_prop_id": [train_df["prop_id"].nunique(), test_df["prop_id"].nunique()],
        "min_date": [train_df["date_time"].min(), test_df["date_time"].min()],
        "max_date": [train_df["date_time"].max(), test_df["date_time"].max()],
    }
).set_index("split")

display(schema_summary)
print("Train-only columns:", train_only_cols)
print("Test-only columns:", test_only_cols)

train_only_usage = pd.DataFrame(
    [
        {
            "column": "position",
            "safe_use": "EDA only",
            "why_not_feature": "Not available in test; strongly affected by Expedia's previous ranking",
        },
        {
            "column": "click_bool",
            "safe_use": "Target/relevance construction and validation",
            "why_not_feature": "This is part of the answer we are trying to predict",
        },
        {
            "column": "booking_bool",
            "safe_use": "Target/relevance construction and validation",
            "why_not_feature": "This is the strongest target signal",
        },
        {
            "column": "gross_bookings_usd",
            "safe_use": "EDA on booked rows only",
            "why_not_feature": "Only known after booking; unavailable at prediction time",
        },
    ]
)

display(train_only_usage)


,rows,columns,unique_srch_id,unique_prop_id,min_date,max_date
split,,,,,,
train,4958347,54,199795,129113,2012-11-01 00:08:29,2013-06-30 23:58:24
test,4959183,50,199549,129438,2012-11-01 00:01:37,2013-06-30 23:55:44


Train-only columns: ['booking_bool', 'click_bool', 'gross_bookings_usd', 'position']
Test-only columns: []


,column,safe_use,why_not_feature
0,position,EDA only,Not available in test; strongly affected by Ex...
1,click_bool,Target/relevance construction and validation,This is part of the answer we are trying to pr...
2,booking_bool,Target/relevance construction and validation,This is the strongest target signal
3,gross_bookings_usd,EDA on booked rows only,Only known after booking; unavailable at predi...


In [5]:
dtype_table = pd.DataFrame(
    {
        "train_dtype": train_df.dtypes.astype(str),
        "test_dtype": test_df.dtypes.astype(str).reindex(train_df.columns),
        "train_null_pct": train_df.isna().mean().mul(100),
        "test_null_pct": test_df.isna().mean().mul(100).reindex(train_df.columns),
    }
).sort_values(["train_null_pct", "test_null_pct"], ascending=False)

display(dtype_table)


,train_dtype,test_dtype,train_null_pct,test_null_pct
comp1_rate_percent_diff,float32,float32,98.09535,98.17575
comp6_rate_percent_diff,float32,float32,98.06036,98.04125
comp1_rate,Int8,Int8,97.58125,97.66341
comp1_inv,Int8,Int8,97.38705,97.48196
comp4_rate_percent_diff,float32,float32,97.35626,97.31555
gross_bookings_usd,float32,NaN,97.20895,NaN
comp7_rate_percent_diff,float32,float32,97.20643,97.19061
comp6_rate,Int8,Int8,95.15651,95.11351
visitor_hist_starrating,float32,float32,94.92036,94.88966
visitor_hist_adr_usd,float32,float32,94.89774,94.86561


## 2. Target and Relevance Diagnostics

NDCG@5 cares about ranking relevant hotels near the top of each `srch_id`. Bookings are much more valuable than clicks.


In [6]:
train_df["relevance"] = 0
train_df.loc[train_df["click_bool"] == 1, "relevance"] = 1
train_df.loc[train_df["booking_bool"] == 1, "relevance"] = 5

train_df["relevance"] = train_df["relevance"].astype("int8")

label_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "clicked rows",
            "booked rows",
            "clicked-only rows",
            "non-relevant rows",
            "click rate",
            "booking rate",
            "relevance mean",
        ],
        "value": [
            len(train_df),
            int(train_df["click_bool"].sum()),
            int(train_df["booking_bool"].sum()),
            int(
                ((train_df["click_bool"] == 1) & (train_df["booking_bool"] == 0)).sum()
            ),
            int((train_df["relevance"] == 0).sum()),
            train_df["click_bool"].mean(),
            train_df["booking_bool"].mean(),
            train_df["relevance"].mean(),
        ],
    }
)
print("Label summary:")
display(label_summary)

counts = train_df["relevance"].value_counts(dropna=False)

print("Relevance value counts and percentages:")
display(pd.DataFrame({"rows": counts, "row_pct": counts / len(train_df) * 100}))


Label summary:


,metric,value
0,rows,4958347.00000
1,clicked rows,221879.00000
2,booked rows,138390.00000
3,clicked-only rows,83489.00000
4,non-relevant rows,4736468.00000
5,click rate,0.04475
6,booking rate,0.02791
7,relevance mean,0.15639


Relevance value counts and percentages:


,rows,row_pct
relevance,,
0,4736468,95.52514
5,138390,2.79105
1,83489,1.68381


In [7]:
search_labels = train_df.groupby("srch_id", observed=True).agg(
    candidates=("prop_id", "size"),
    clicks=("click_bool", "sum"),
    bookings=("booking_bool", "sum"),
    max_relevance=("relevance", "max"),
    random_bool=("random_bool", "max"),
)
search_labels["clicked_no_booking"] = (
    search_labels["clicks"] - search_labels["bookings"]
)
search_labels["has_click"] = search_labels["clicks"].gt(0)
search_labels["has_booking"] = search_labels["bookings"].gt(0)
search_labels["has_any_relevance"] = search_labels["max_relevance"].gt(0)

query_summary = pd.DataFrame(
    {
        "metric": [
            "searches",
            "median candidates/search",
            "mean candidates/search",
            "max candidates/search",
            "searches with any click",
            "searches with booking",
            "searches with no positive label",
            "randomized searches pct",
        ],
        "value": [
            len(search_labels),
            search_labels["candidates"].median(),
            search_labels["candidates"].mean(),
            search_labels["candidates"].max(),
            search_labels["has_click"].sum(),
            search_labels["has_booking"].sum(),
            (~search_labels["has_any_relevance"]).sum(),
            search_labels["random_bool"].mean(),
        ],
    }
)
display(query_summary)

display(
    search_labels[["candidates", "clicks", "bookings", "clicked_no_booking"]]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T
)


,metric,value
0,searches,199795.00000
1,median candidates/search,29.00000
2,mean candidates/search,24.81717
3,max candidates/search,38.00000
4,searches with any click,199795.00000
5,searches with booking,138390.00000
6,searches with no positive label,0.00000
7,randomized searches pct,0.30407


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
candidates,199795.00000,24.81717,9.11334,5.00000,5.00000,7.00000,18.00000,29.00000,32.00000,34.00000,35.00000,38.00000
clicks,199795.00000,1.11053,0.56339,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,2.00000,3.00000,25.00000
bookings,199795.00000,0.69266,0.46139,0.00000,0.00000,0.00000,0.00000,1.00000,1.00000,1.00000,1.00000,1.00000
clicked_no_booking,199795.00000,0.41787,0.73655,0.00000,0.00000,0.00000,0.00000,0.00000,1.00000,1.00000,3.00000,25.00000


In [8]:
# Create one label per search group
search_labels = search_labels.copy()

search_labels["query_label_pattern"] = "no_positive_label"

search_labels.loc[search_labels["clicks"] > 0, "query_label_pattern"] = "click_only"

search_labels.loc[search_labels["bookings"] > 0, "query_label_pattern"] = "has_booking"

# Count how many searches fall into each pattern
query_label_patterns = (
    search_labels["query_label_pattern"].value_counts().to_frame(name="searches")
)

# Add percentage of total searches
query_label_patterns["pct_searches"] = query_label_patterns["searches"] / len(
    search_labels
)

display(query_label_patterns)

# Candidate count distribution matters because NDCG is computed within complete search groups.
candidate_count_dist = (
    search_labels["candidates"]
    .value_counts()
    .sort_index()
    .rename_axis("candidates")
    .to_frame("searches")
)
candidate_count_dist["pct_searches"] = candidate_count_dist["searches"] / len(
    search_labels
)
display(candidate_count_dist)


,searches,pct_searches
query_label_pattern,,
has_booking,138390,0.69266
click_only,61405,0.30734


,searches,pct_searches
candidates,,
5,4460,0.02232
6,4409,0.02207
7,4066,0.02035
8,4003,0.02004
9,3923,0.01964
10,3756,0.01880
11,3688,0.01846
12,3624,0.01814
13,3626,0.01815


## 3. Position Bias Analysis

`position` exists only in train. We Use this to understand historical display bias and randomized rows, but we do not use it as a final prediction feature.


In [9]:
position_summary = (
    train_df.groupby("position", observed=True)
    .agg(
        rows=("prop_id", "size"),
        click_rate=("click_bool", "mean"),
        booking_rate=("booking_bool", "mean"),
        relevance_mean=("relevance", "mean"),
        randomized_pct=("random_bool", "mean"),
    )
    .sort_index()
)
position_summary["row_pct"] = position_summary["rows"] / len(train_df)
display(position_summary)

position_by_random = (
    train_df.groupby(["random_bool", "position"], observed=True)
    .agg(
        rows=("prop_id", "size"),
        click_rate=("click_bool", "mean"),
        booking_rate=("booking_bool", "mean"),
        relevance_mean=("relevance", "mean"),
    )
    .reset_index()
)
display(position_by_random)


,rows,click_rate,booking_rate,relevance_mean,randomized_pct,row_pct
position,,,,,,
1,199415,0.19251,0.14099,0.75646,0.30308,0.04022
2,199350,0.13423,0.09427,0.51131,0.30286,0.04020
3,199355,0.10528,0.07083,0.38860,0.30301,0.04021
4,193406,0.08820,0.05786,0.31963,0.31115,0.03901
5,9312,0.02255,0.01160,0.06894,0.10535,0.00188
6,199172,0.07141,0.04517,0.25209,0.30307,0.04017
7,194769,0.06053,0.03789,0.21207,0.30205,0.03928
8,190375,0.05292,0.03237,0.18239,0.30141,0.03839
9,186236,0.04623,0.02748,0.15613,0.30089,0.03756


,random_bool,position,rows,click_rate,booking_rate,relevance_mean
0,0,1,138977,0.21520,0.19438,0.99271
1,0,2,138974,0.14586,0.12935,0.66325
2,0,3,138949,0.11070,0.09633,0.49602
3,0,4,133228,0.09208,0.07951,0.41012
4,0,5,8331,0.01789,0.01176,0.06494
...,...,...,...,...,...,...
75,1,36,17379,0.01772,0.00144,0.02348
76,1,37,9076,0.01730,0.00132,0.02259
77,1,38,2426,0.01319,0.00165,0.01979
78,1,39,393,0.01781,0.00509,0.03817


## 4. Missingness and Data Quality

Missingness is a strong feature family here, especially visitor history, query affinity, distance, and competitor blocks.


In [10]:
def missingness_report(train, test, columns=None, top_n=80):
    if columns is None:
        columns = sorted(set(train.columns) | set(test.columns))
    out = pd.DataFrame(index=columns)
    out["train_missing_pct"] = train.reindex(columns=columns).isna().mean().mul(100)
    out["test_missing_pct"] = test.reindex(columns=columns).isna().mean().mul(100)
    out["abs_diff_pct"] = (out["train_missing_pct"] - out["test_missing_pct"]).abs()
    return out.sort_values(["abs_diff_pct", "train_missing_pct", "test_missing_pct"], ascending=False).head(top_n)

miss = missingness_report(train_df, test_df, columns=common_cols)
display(miss)

row_missing = pd.DataFrame({
    "split": ["train", "test"],
    "mean_missing_per_row": [train_df[common_cols].isna().sum(axis=1).mean(), test_df[common_cols].isna().sum(axis=1).mean()],
    "median_missing_per_row": [train_df[common_cols].isna().sum(axis=1).median(), test_df[common_cols].isna().sum(axis=1).median()],
    "p95_missing_per_row": [train_df[common_cols].isna().sum(axis=1).quantile(.95), test_df[common_cols].isna().sum(axis=1).quantile(.95)],
})
display(row_missing)


,train_missing_pct,test_missing_pct,abs_diff_pct
comp8_inv,59.91602,60.21754,0.30152
comp8_rate,61.34490,61.63906,0.29416
comp3_inv,66.70281,66.90521,0.20240
comp3_rate,69.05646,69.24927,0.19281
comp2_inv,57.03671,57.22543,0.18872
comp2_rate,59.16639,59.34893,0.18254
comp4_rate,93.80080,93.69410,0.10669
comp4_inv,93.06900,92.96642,0.10258
comp1_inv,97.38705,97.48196,0.09491
comp1_rate,97.58125,97.66341,0.08216


,split,mean_missing_per_row,median_missing_per_row,p95_missing_per_row
0,train,23.17573,23.00000,28.00000
1,test,23.18895,23.00000,28.00000


In [11]:
comp_cols = [c for c in common_cols if c.startswith("comp")]
rate_cols = [c for c in comp_cols if c.endswith("_rate")]
inv_cols = [c for c in comp_cols if c.endswith("_inv")]
pct_diff_cols = [c for c in comp_cols if c.endswith("_rate_percent_diff")]

comp_summary = pd.DataFrame(
    {
        "feature_type": ["rate", "inventory", "rate_percent_diff"],
        "columns": [len(rate_cols), len(inv_cols), len(pct_diff_cols)],
        "train_missing_pct_mean": [
            train_df[rate_cols].isna().mean().mean() * 100,
            train_df[inv_cols].isna().mean().mean() * 100,
            train_df[pct_diff_cols].isna().mean().mean() * 100,
        ],
        "test_missing_pct_mean": [
            test_df[rate_cols].isna().mean().mean() * 100,
            test_df[inv_cols].isna().mean().mean() * 100,
            test_df[pct_diff_cols].isna().mean().mean() * 100,
        ],
    }
)
display(comp_summary)

comp_eda = train_df[["relevance", "click_bool", "booking_bool"]].copy()
comp_eda["comp_rate_missing_count"] = train_df[rate_cols].isna().sum(axis=1)
comp_eda["comp_inv_missing_count"] = train_df[inv_cols].isna().sum(axis=1)
comp_eda["comp_pct_diff_missing_count"] = train_df[pct_diff_cols].isna().sum(axis=1)

for c in rate_cols:
    # In Expedia data, negative / positive rate values encode whether Expedia is cheaper / more expensive than a competitor.
    comp_eda[f"{c}_neg"] = train_df[c].lt(0)
    comp_eda[f"{c}_pos"] = train_df[c].gt(0)
comp_eda["competitors_with_lower_rate"] = comp_eda[[f"{c}_neg" for c in rate_cols]].sum(
    axis=1
)
comp_eda["competitors_with_higher_rate"] = comp_eda[
    [f"{c}_pos" for c in rate_cols]
].sum(axis=1)

display(
    comp_eda.groupby("competitors_with_lower_rate", observed=True).agg(
        rows=("relevance", "size"),
        click_rate=("click_bool", "mean"),
        booking_rate=("booking_bool", "mean"),
        relevance_mean=("relevance", "mean"),
    )
)
display(
    comp_eda.groupby("competitors_with_higher_rate", observed=True).agg(
        rows=("relevance", "size"),
        click_rate=("click_bool", "mean"),
        booking_rate=("booking_bool", "mean"),
        relevance_mean=("relevance", "mean"),
    )
)


,feature_type,columns,train_missing_pct_mean,test_missing_pct_mean
0,rate,8,78.11569,78.19219
1,inventory,8,76.75787,76.83640
2,rate_percent_diff,8,92.57545,92.60019


,rows,click_rate,booking_rate,relevance_mean
competitors_with_lower_rate,,,,
0,4370521,0.04558,0.02835,0.15898
1,315568,0.04095,0.02576,0.14400
2,176908,0.03614,0.02330,0.12934
3,72239,0.03598,0.02398,0.13188
4,21688,0.03283,0.02144,0.11859
5,1225,0.03347,0.02041,0.11510
6,198,0.04040,0.04040,0.20202


,rows,click_rate,booking_rate,relevance_mean
competitors_with_higher_rate,,,,
0,4176594,0.04359,0.02671,0.15044
1,521217,0.04823,0.03175,0.17522
2,170549,0.05458,0.03754,0.20476
3,65429,0.05828,0.04183,0.22560
4,22565,0.06519,0.04733,0.25451
5,1678,0.05244,0.03218,0.18117
6,315,0.04762,0.03175,0.17460


## 5. Cardinality and Train/Test Coverage

High-cardinality IDs are useful for popularity and target encoding, but label-derived encodings must be out-of-fold for validation.


In [12]:
id_like_cols = [
    "site_id",
    "visitor_location_country_id",
    "prop_country_id",
    "prop_id",
    "srch_destination_id",
    "hotel_market",
]
id_like_cols = [c for c in id_like_cols if c in common_cols]

coverage_rows = []
for c in id_like_cols:
    tr_values = set(train_df[c].dropna().unique())
    te_values = set(test_df[c].dropna().unique())
    coverage_rows.append(
        {
            "column": c,
            "train_unique": len(tr_values),
            "test_unique": len(te_values),
            "test_unseen_in_train": len(te_values - tr_values),
            "test_unseen_pct": len(te_values - tr_values) / max(len(te_values), 1),
            "train_absent_in_test": len(tr_values - te_values),
        }
    )
coverage = pd.DataFrame(coverage_rows).sort_values("test_unseen_pct", ascending=False)
display(coverage)

cardinality = []
for c in common_cols:
    nunique_train = train_df[c].nunique(dropna=True)
    nunique_test = test_df[c].nunique(dropna=True)
    cardinality.append((c, str(train_df[c].dtype), nunique_train, nunique_test))
cardinality = pd.DataFrame(
    cardinality, columns=["column", "dtype", "train_unique", "test_unique"]
)
display(cardinality.sort_values("train_unique", ascending=False))


,column,train_unique,test_unique,test_unseen_in_train,test_unseen_pct,train_absent_in_test
4,srch_destination_id,18127,18049,5588,0.30960,5666
3,prop_id,129113,129438,7773,0.06005,7448
1,visitor_location_country_id,210,210,8,0.03810,8
0,site_id,34,34,0,0.00000,0
2,prop_country_id,172,167,0,0.00000,5


,column,dtype,train_unique,test_unique
25,orig_destination_distance,float32,530595,534879
42,srch_id,int32,199795,199549
44,srch_query_affinity_score,float32,199387,201787
24,date_time,datetime64[us],198615,198338
30,prop_id,int32,129113,129438
26,price_usd,float32,76465,76538
41,srch_destination_id,int32,18127,18049
32,prop_location_score2,float32,9342,9357
47,visitor_hist_adr_usd,float32,7799,7843
2,comp1_rate_percent_diff,float32,1830,1679


## 6. Numeric Distribution and Drift Checks

Large train/test drift can explain leaderboard instability. This section gives both direct quantiles and compact drift scores.


In [13]:
core_numeric_cols = [
    "price_usd",
    "prop_starrating",
    "prop_review_score",
    "prop_location_score1",
    "prop_location_score2",
    "prop_log_historical_price",
    "promotion_flag",
    "srch_length_of_stay",
    "srch_booking_window",
    "srch_adults_count",
    "srch_children_count",
    "srch_room_count",
    "srch_saturday_night_bool",
    "srch_query_affinity_score",
    "orig_destination_distance",
    "visitor_hist_starrating",
    "visitor_hist_adr_usd",
    "random_bool",
]
core_numeric_cols = [c for c in core_numeric_cols if c in common_cols]

quantiles = [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00]
summary_parts = []
for split, df in [("train", train_df), ("test", test_df)]:
    q = df[core_numeric_cols].quantile(quantiles).T
    q.columns = [f"q{int(p * 100):02d}" for p in quantiles]
    q.insert(0, "split", split)
    q["mean"] = df[core_numeric_cols].mean()
    q["missing_pct"] = df[core_numeric_cols].isna().mean().mul(100)
    summary_parts.append(q.reset_index(names="column"))
num_summary = pd.concat(summary_parts, ignore_index=True)
display(num_summary.sort_values(["column", "split"]))


,column,split,q00,q01,q05,q25,q50,q75,q95,q99,q100,mean,missing_pct
32,orig_destination_distance,test,0.01000,0.93000,13.19000,142.28000,387.85001,1517.56250,5863.36865,9689.55957,11692.98047,1312.80164,32.43839
14,orig_destination_distance,train,0.01000,0.95000,13.61000,139.80000,386.60001,1500.67004,5828.75781,9677.46973,11666.63965,1301.23438,32.42577
18,price_usd,test,0.00000,34.00000,51.00000,85.00000,122.26000,185.00000,356.01999,599.00000,9661340.00000,229.35754,0.00000
0,price_usd,train,0.00000,34.00000,51.00000,85.00000,122.00000,184.96001,356.00000,599.00000,19726328.00000,254.20959,0.00000
24,promotion_flag,test,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,1.00000,1.00000,1.00000,0.21590,0.00000
6,promotion_flag,train,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,1.00000,1.00000,1.00000,0.21562,0.00000
21,prop_location_score1,test,0.00000,0.00000,0.00000,1.79000,2.77000,4.04000,5.55000,6.28000,6.98000,2.87937,0.00000
3,prop_location_score1,train,0.00000,0.00000,0.00000,1.79000,2.77000,4.04000,5.54000,6.28000,6.98000,2.87259,0.00000
22,prop_location_score2,test,0.00000,0.00030,0.00190,0.01900,0.06920,0.18070,0.47940,0.72055,1.00000,0.13045,21.93974
4,prop_location_score2,train,0.00000,0.00030,0.00190,0.01900,0.06900,0.18050,0.47920,0.71800,1.00000,0.13039,21.99015


In [14]:
def numeric_drift_report(train, test, columns, sample_n=500_000, random_state=RANDOM_STATE):
    train_sample = train.sample(n=min(sample_n, len(train)), random_state=random_state) if len(train) > sample_n else train
    test_sample = test.sample(n=min(sample_n, len(test)), random_state=random_state) if len(test) > sample_n else test
    rows = []
    for c in columns:
        tr = train_sample[c]
        te = test_sample[c]
        tr_mean, te_mean = tr.mean(), te.mean()
        tr_std, te_std = tr.std(), te.std()
        pooled = np.sqrt((tr_std**2 + te_std**2) / 2) if pd.notna(tr_std) and pd.notna(te_std) else np.nan
        rows.append({
            "column": c,
            "train_mean": tr_mean,
            "test_mean": te_mean,
            "mean_diff": te_mean - tr_mean,
            "standardized_mean_diff": (te_mean - tr_mean) / pooled if pooled and pooled > 0 else np.nan,
            "train_missing_pct": tr.isna().mean() * 100,
            "test_missing_pct": te.isna().mean() * 100,
            "missing_diff_pct": (te.isna().mean() - tr.isna().mean()) * 100,
        })
    return pd.DataFrame(rows).sort_values("standardized_mean_diff", key=lambda s: s.abs(), ascending=False)

drift = numeric_drift_report(train_df, test_df, core_numeric_cols)
display(drift)


,column,train_mean,test_mean,mean_diff,standardized_mean_diff,train_missing_pct,test_missing_pct,missing_diff_pct
13,srch_query_affinity_score,-24.05913,-24.46314,-0.40401,-0.02571,93.62400,93.56660,-0.05740
12,srch_saturday_night_bool,0.50223,0.49868,-0.00356,-0.00711,0.00000,0.00000,0.00000
3,prop_location_score1,2.87385,2.88343,0.00958,0.00625,0.00000,0.00000,0.00000
14,orig_destination_distance,1297.58496,1309.81458,12.22961,0.00604,32.40540,32.50000,0.09460
15,visitor_hist_starrating,3.37839,3.37440,-0.00400,-0.00578,94.90000,94.88460,-0.01540
10,srch_children_count,0.34987,0.34605,-0.00383,-0.00525,0.00000,0.00000,0.00000
16,visitor_hist_adr_usd,176.55316,176.08424,-0.46892,-0.00433,94.87960,94.86240,-0.01720
0,price_usd,256.68353,217.20126,-39.48227,-0.00391,0.00000,0.00000,0.00000
8,srch_booking_window,37.51375,37.70133,0.18758,0.00360,0.00000,0.00000,0.00000
6,promotion_flag,0.21565,0.21675,0.00110,0.00266,0.00000,0.00000,0.00000


In [15]:
# Price often contains extreme values. Inspect after clipping/logging before modeling.
price_quality = pd.DataFrame(
    {
        "split": ["train", "test"],
        "non_positive_price_rows": [
            train_df["price_usd"].le(0).sum(),
            test_df["price_usd"].le(0).sum(),
        ],
        "price_gt_1000_rows": [
            train_df["price_usd"].gt(1000).sum(),
            test_df["price_usd"].gt(1000).sum(),
        ],
        "price_gt_5000_rows": [
            train_df["price_usd"].gt(5000).sum(),
            test_df["price_usd"].gt(5000).sum(),
        ],
        "price_missing_rows": [
            train_df["price_usd"].isna().sum(),
            test_df["price_usd"].isna().sum(),
        ],
    }
)
display(price_quality)

display(
    train_df.nlargest(20, "price_usd")[
        [
            "srch_id",
            "prop_id",
            "price_usd",
            "prop_starrating",
            "prop_review_score",
            "click_bool",
            "booking_bool",
            "position",
        ]
    ]
)


,split,non_positive_price_rows,price_gt_1000_rows,price_gt_5000_rows,price_missing_rows
0,train,31,13314,2871,0
1,test,23,13704,2836,0


,srch_id,prop_id,price_usd,prop_starrating,prop_review_score,click_bool,booking_bool,position
1168566,78107,39677,19726328.00000,5,5.00000,0,0,12
680748,45559,114295,11818011.00000,4,0.00000,0,0,14
3117007,209314,130892,9381309.00000,5,5.00000,0,0,1
1168574,78107,81654,5444467.00000,5,4.50000,0,0,9
2945135,197817,17580,4973355.00000,4,4.00000,0,0,1
1168576,78107,87788,4884239.00000,5,4.50000,0,0,3
1168562,78107,7077,4339792.00000,5,4.50000,0,0,27
1168580,78107,101697,4260887.00000,5,4.50000,0,0,4
4172824,279943,54596,4216286.00000,4,4.00000,0,0,20
1168581,78107,109731,3905813.00000,5,5.00000,0,0,2


## 7. Time and Search Intent

Use `date_time`, booking window, stay length, party size, rooms, and weekend flags to understand demand segments.


In [16]:
for df in (train_df, test_df):
    df["search_month"] = df["date_time"].dt.month.astype("int8")
    df["search_dow"] = df["date_time"].dt.dayofweek.astype("int8")
    df["search_hour"] = df["date_time"].dt.hour.astype("int8")
    df["party_size"] = (df["srch_adults_count"] + df["srch_children_count"]).astype(
        "int16"
    )
    df["rooms_per_person"] = df["srch_room_count"] / df["party_size"].replace(0, np.nan)

for c in [
    "search_month",
    "search_dow",
    "search_hour",
    "srch_length_of_stay",
    "srch_booking_window",
    "party_size",
    "srch_room_count",
    "srch_saturday_night_bool",
]:
    if c in train_df.columns:
        tmp = train_df.groupby(c, observed=True).agg(
            rows=("srch_id", "size"),
            click_rate=("click_bool", "mean"),
            booking_rate=("booking_bool", "mean"),
            relevance_mean=("relevance", "mean"),
        )
        tmp["row_pct"] = tmp["rows"] / len(train_df)
        print(f"\nTarget by {c}")
        display(tmp)



Target by search_month


,rows,click_rate,booking_rate,relevance_mean,row_pct
search_month,,,,,
1,607578,0.04377,0.02635,0.14918,0.12254
2,593893,0.04434,0.02774,0.15529,0.11978
3,700876,0.04476,0.02772,0.15562,0.14135
4,642877,0.04504,0.02792,0.15673,0.12966
5,699776,0.04566,0.02869,0.16043,0.14113
6,737391,0.04509,0.02863,0.15959,0.14872
11,496217,0.04475,0.02816,0.15737,0.10008
12,479739,0.04423,0.02787,0.15572,0.09675



Target by search_dow


,rows,click_rate,booking_rate,relevance_mean,row_pct
search_dow,,,,,
0,769706,0.04473,0.02769,0.15549,0.15523
1,768196,0.04468,0.02760,0.15509,0.15493
2,784298,0.04459,0.02797,0.15648,0.15818
3,745478,0.04475,0.02810,0.15716,0.15035
4,681372,0.04454,0.02840,0.15815,0.13742
5,548477,0.04505,0.02810,0.15746,0.11062
6,660820,0.04500,0.02757,0.15527,0.13327



Target by search_hour


...

,rows,click_rate,booking_rate,relevance_mean,row_pct
party_size,,,,,
1,906888,0.04253,0.03029,0.16370,0.18290
2,2721200,0.04467,0.02708,0.15297,0.54881
3,570225,0.04523,0.02803,0.15737,0.11500
4,547807,0.04586,0.02736,0.15528,0.11048
5,104253,0.05214,0.02987,0.17162,0.02103
6,61042,0.05019,0.02952,0.16828,0.01231
7,17246,0.05074,0.02928,0.16787,0.00348
8,19962,0.04804,0.02936,0.16546,0.00403
9,4314,0.05424,0.02619,0.15902,0.00087



Target by srch_room_count


,rows,click_rate,booking_rate,relevance_mean,row_pct
srch_room_count,,,,,
1,4520212,0.04415,0.02735,0.15354,0.91164
2,369152,0.05157,0.03444,0.18931,0.07445
3,47656,0.04669,0.03022,0.16755,0.00961
4,12175,0.04903,0.03006,0.16928,0.00246
5,3829,0.05171,0.03369,0.18647,0.00077
6,2181,0.04310,0.02705,0.15131,0.00044
7,1183,0.04480,0.02451,0.14286,0.00024
8,1959,0.04952,0.02399,0.14548,0.00040



Target by srch_saturday_night_bool


,rows,click_rate,booking_rate,relevance_mean,row_pct
srch_saturday_night_bool,,,,,
0,2468202,0.04451,0.02700,0.15253,0.49779
1,2490145,0.04499,0.02881,0.16022,0.50221


## 8. Search-relative Feature Diagnostics

MUST DO for ranking models: compare each hotel to competitors in the same `srch_id`. These features are available in train and test because they do not use labels.


In [17]:
relative_cols = ["price_usd", "prop_starrating", "prop_review_score", "prop_location_score1", "prop_location_score2", "prop_log_historical_price"]
relative_cols = [c for c in relative_cols if c in common_cols]

# Build a compact working frame to avoid duplicating the full dataset.
rel = train_df[["srch_id", "prop_id", "click_bool", "booking_bool", "relevance", "promotion_flag"] + relative_cols].copy()

g = rel.groupby("srch_id", observed=True)
for c in relative_cols:
    rel[f"{c}_rank_pct_asc"] = g[c].rank(method="average", pct=True, ascending=True)
    rel[f"{c}_rank_pct_desc"] = g[c].rank(method="average", pct=True, ascending=False)
    rel[f"{c}_search_mean"] = g[c].transform("mean")
    rel[f"{c}_search_median"] = g[c].transform("median")
    rel[f"{c}_diff_mean"] = rel[c] - rel[f"{c}_search_mean"]
    rel[f"{c}_diff_median"] = rel[c] - rel[f"{c}_search_median"]

rel["log_price_usd"] = np.log1p(rel["price_usd"].clip(lower=0))
rel["value_star_per_log_price"] = rel["prop_starrating"] / rel["log_price_usd"].replace(0, np.nan)
rel["value_review_per_log_price"] = rel["prop_review_score"] / rel["log_price_usd"].replace(0, np.nan)

def target_by_quantile(df, feature, q=10):
    s = df[feature]
    try:
        bucket = pd.qcut(s, q=q, duplicates="drop")
    except ValueError:
        bucket = pd.cut(s, bins=q)
    return (
        df.assign(bucket=bucket)
        .groupby("bucket", observed=True)
        .agg(rows=("relevance", "size"), click_rate=("click_bool", "mean"), booking_rate=("booking_bool", "mean"), relevance_mean=("relevance", "mean"))
        .assign(row_pct=lambda d: d["rows"] / len(df))
    )

for c in [
    "price_usd_rank_pct_asc", "price_usd_diff_median", "prop_starrating_rank_pct_desc",
    "prop_review_score_rank_pct_desc", "prop_location_score1_rank_pct_desc",
    "value_star_per_log_price", "value_review_per_log_price",
]:
    if c in rel.columns:
        print(f"\nTarget by quantile: {c}")
        display(target_by_quantile(rel, c))

print("\nTarget by promotion flag")
display(rel.groupby("promotion_flag", observed=True).agg(rows=("relevance", "size"), click_rate=("click_bool", "mean"), booking_rate=("booking_bool", "mean"), relevance_mean=("relevance", "mean")))



Target by quantile: price_usd_rank_pct_asc


,rows,click_rate,booking_rate,relevance_mean,row_pct
bucket,,,,,
"(0.0253, 0.121]",499887,0.05619,0.03565,0.19877,0.10082
"(0.121, 0.219]",495489,0.05591,0.03555,0.19811,0.09993
"(0.219, 0.321]",494853,0.05018,0.03208,0.17850,0.09980
"(0.321, 0.421]",495498,0.04977,0.03220,0.17858,0.09993
"(0.421, 0.517]",496316,0.04642,0.02971,0.16525,0.10010
"(0.517, 0.621]",493923,0.04478,0.02820,0.15760,0.09961
"(0.621, 0.719]",500275,0.04141,0.02568,0.14412,0.10090
"(0.719, 0.818]",499862,0.03843,0.02333,0.13177,0.10081
"(0.818, 0.92]",489242,0.03479,0.02053,0.11691,0.09867



Target by quantile: price_usd_diff_median


,rows,click_rate,booking_rate,relevance_mean,row_pct
bucket,,,,,
"(-1199360.001, -49.93]",495861,0.06265,0.03782,0.21391,0.10001
"(-49.93, -30.0]",520739,0.05082,0.03278,0.18195,0.10502
"(-30.0, -18.0]",488468,0.04923,0.03217,0.17792,0.09851
"(-18.0, -8.0]",486677,0.04956,0.03250,0.17954,0.09815
"(-8.0, 0.0]",586880,0.04929,0.03181,0.17654,0.11836
"(0.0, 8.85]",396433,0.04404,0.02826,0.15709,0.07995
"(8.85, 21.09]",495808,0.04026,0.02560,0.14267,0.09999
"(21.09, 42.0]",501402,0.03867,0.02392,0.13433,0.10112
"(42.0, 85.0]",493869,0.03478,0.02052,0.11685,0.09960



Target by quantile: prop_starrating_rank_pct_desc


...

,rows,click_rate,booking_rate,relevance_mean,row_pct
bucket,,,,,
"(-0.001, 0.444]",499850,0.03146,0.01815,0.10407,0.10081
"(0.444, 0.509]",501528,0.03363,0.02264,0.12419,0.10115
"(0.509, 0.591]",499353,0.03485,0.01996,0.11470,0.10071
"(0.591, 0.627]",483580,0.03948,0.02454,0.13764,0.09753
"(0.627, 0.66]",502396,0.04422,0.02899,0.16016,0.10132
"(0.66, 0.701]",490859,0.04554,0.02949,0.16350,0.09900
"(0.701, 0.753]",493262,0.04669,0.02807,0.15897,0.09948
"(0.753, 0.805]",498745,0.04996,0.03033,0.17126,0.10059
"(0.805, 0.87]",493520,0.05867,0.03728,0.20780,0.09953



Target by quantile: value_review_per_log_price


,rows,click_rate,booking_rate,relevance_mean,row_pct
bucket,,,,,
"(-0.001, 0.61]",495102,0.02859,0.01346,0.08242,0.09985
"(0.61, 0.706]",497676,0.03637,0.01981,0.11560,0.10037
"(0.706, 0.752]",493221,0.04051,0.02339,0.13405,0.09947
"(0.752, 0.787]",497922,0.04194,0.02481,0.14118,0.10042
"(0.787, 0.818]",492736,0.04497,0.02751,0.15502,0.09938
"(0.818, 0.849]",505475,0.04685,0.02971,0.16568,0.10194
"(0.849, 0.88]",483584,0.04818,0.03132,0.17348,0.09753
"(0.88, 0.919]",495661,0.05058,0.03366,0.18524,0.09996
"(0.919, 0.973]",496254,0.05284,0.03599,0.19679,0.10008



Target by promotion flag


,rows,click_rate,booking_rate,relevance_mean
promotion_flag,,,,
0,3889229,0.04047,0.02480,0.13966
1,1069118,0.06030,0.03924,0.21724


## 9. Historical Popularity and Target Signal

HIGH IMPACT: property, destination, and property-by-destination historical relevance. These are label-derived, so use out-of-fold encodings for validation and fit on full train only for final test inference.


In [18]:
def label_aggregate(df, keys, min_rows=50):
    out = (
        df.groupby(keys, observed=True)
        .agg(
            rows=("srch_id", "size"),
            searches=("srch_id", "nunique"),
            click_rate=("click_bool", "mean"),
            booking_rate=("booking_bool", "mean"),
            relevance_mean=("relevance", "mean"),
            avg_position=("position", "mean"),
            avg_price=("price_usd", "mean"),
        )
        .reset_index()
    )
    return out[out["rows"].ge(min_rows)].sort_values(["relevance_mean", "rows"], ascending=False)

prop_pop = label_aggregate(train_df, ["prop_id"], min_rows=100)
dest_pop = label_aggregate(train_df, ["srch_destination_id"], min_rows=500)
prop_dest_pop = label_aggregate(train_df, ["prop_id", "srch_destination_id"], min_rows=30)

print("Top properties by historical relevance, min 100 rows")
display(prop_pop.head(30))
print("Top destinations by historical relevance, min 500 rows")
display(dest_pop.head(30))
print("Top property x destination combinations by historical relevance, min 30 rows")
display(prop_dest_pop.head(30))

# Coverage of useful encodings on test.
for name, keys, agg in [
    ("prop_id", ["prop_id"], prop_pop),
    ("srch_destination_id", ["srch_destination_id"], dest_pop),
    ("prop_id x srch_destination_id", ["prop_id", "srch_destination_id"], prop_dest_pop),
]:
    train_keys = set(map(tuple, agg[keys].to_numpy())) if len(keys) > 1 else set(agg[keys[0]])
    test_keys_raw = test_df[keys].drop_duplicates()
    test_keys = set(map(tuple, test_keys_raw.to_numpy())) if len(keys) > 1 else set(test_keys_raw[keys[0]])
    print(f"{name}: test key coverage from min-count aggregate = {len(test_keys & train_keys) / max(len(test_keys), 1):.3%}")


Top properties by historical relevance, min 100 rows


,prop_id,rows,searches,click_rate,booking_rate,relevance_mean,avg_position,avg_price
60967,66456,109,109,0.44954,0.41284,2.10092,3.95413,58.91596
90599,98803,314,314,0.36306,0.27070,1.44586,5.69108,12459.06250
9616,10429,149,149,0.32886,0.25503,1.34899,3.11409,177.23148
23457,25517,136,136,0.27206,0.25000,1.27206,10.11029,144.56146
4180,4548,162,162,0.24074,0.21605,1.10494,8.33951,173.80791
45262,49310,221,221,0.26697,0.20362,1.08145,4.94118,231.86815
28244,30713,144,144,0.24306,0.20833,1.07639,6.45833,301.45178
87940,95914,191,191,0.25654,0.20419,1.07330,8.69110,180.11978
88111,96101,111,111,0.24324,0.20721,1.07207,6.67568,88.90144
97791,106652,114,114,0.27193,0.19298,1.04386,10.94737,107.04553


Top destinations by historical relevance, min 500 rows


,srch_destination_id,rows,searches,click_rate,booking_rate,relevance_mean,avg_position,avg_price
13221,20757,1146,125,0.12216,0.08726,0.47120,6.13264,142.25468
11067,17330,524,66,0.13168,0.08397,0.46756,5.15076,95.71278
2215,3422,2377,238,0.10644,0.08119,0.43122,6.50694,50.04003
699,1099,785,65,0.08917,0.07006,0.36943,8.08408,177.07857
11668,18321,593,48,0.10118,0.06239,0.35076,8.25295,199.82137
3747,5783,568,52,0.09507,0.06338,0.34859,7.23063,154.38637
6643,10403,1347,100,0.07721,0.06607,0.34150,8.89978,134.20403
17535,27456,990,76,0.09192,0.06162,0.33838,8.82020,100.23005
3517,5431,611,49,0.08511,0.06056,0.32733,8.00327,141.94621
3825,5900,632,55,0.09177,0.05854,0.32595,7.58703,49.38345


Top property x destination combinations by historical relevance, min 30 rows


,prop_id,srch_destination_id,rows,searches,click_rate,booking_rate,relevance_mean,avg_position,avg_price
382990,88359,14182,35,35,0.74286,0.68571,3.48571,1.00000,120.26571
462340,106652,13993,34,34,0.55882,0.50000,2.55882,2.20588,105.78412
234685,54310,27338,73,73,0.54795,0.49315,2.52055,3.53425,117.71314
287693,66456,6284,42,42,0.47619,0.45238,2.28571,2.23810,56.06286
415890,95914,6867,78,78,0.51282,0.42308,2.20513,1.69231,174.95628
452729,104455,8064,30,30,0.46667,0.43333,2.20000,2.30000,136.43967
287695,66456,27986,37,37,0.45946,0.40541,2.08108,4.78378,65.57541
111189,25517,22608,73,73,0.42466,0.39726,2.01370,2.34247,147.75795
37273,8617,17330,45,45,0.48889,0.35556,1.91111,2.44444,116.16334
119624,27525,7512,61,61,0.44262,0.36066,1.88525,1.86885,211.55328


prop_id: test key coverage from min-count aggregate = 9.643%
srch_destination_id: test key coverage from min-count aggregate = 8.848%
prop_id x srch_destination_id: test key coverage from min-count aggregate = 5.904%


In [19]:
# Relationship between historical property popularity and current labels.
# This is an in-sample diagnostic only; do not use these raw full-train encodings for validation scoring.
global_rel = train_df["relevance"].mean()
prop_score = prop_pop[
    ["prop_id", "rows", "relevance_mean", "booking_rate", "click_rate"]
].copy()
prop_score["smoothed_relevance"] = (
    prop_score["relevance_mean"] * prop_score["rows"] + global_rel * 100
) / (prop_score["rows"] + 100)
train_prop_diag = train_df[
    ["prop_id", "relevance", "click_bool", "booking_bool"]
].merge(prop_score[["prop_id", "rows", "smoothed_relevance"]], on="prop_id", how="left")
train_prop_diag["prop_history_bucket"] = pd.qcut(
    train_prop_diag["smoothed_relevance"], q=10, duplicates="drop"
)

display(
    train_prop_diag.groupby("prop_history_bucket", observed=True).agg(
        rows=("relevance", "size"),
        click_rate=("click_bool", "mean"),
        booking_rate=("booking_bool", "mean"),
        relevance_mean=("relevance", "mean"),
        median_prop_rows=("rows", "median"),
    )
)


,rows,click_rate,booking_rate,relevance_mean,median_prop_rows
prop_history_bucket,,,,,
"(0.0177, 0.0719]",267064,0.01013,0.00327,0.02322,302.00000
"(0.0719, 0.0881]",267122,0.01627,0.00607,0.04055,205.00000
"(0.0881, 0.103]",266964,0.02202,0.01091,0.06565,208.00000
"(0.103, 0.119]",266660,0.02829,0.01476,0.08734,205.00000
"(0.119, 0.134]",267251,0.03363,0.01956,0.11188,209.00000
"(0.134, 0.155]",268028,0.04139,0.02445,0.13918,206.00000
"(0.155, 0.179]",265642,0.04880,0.03073,0.17170,251.00000
"(0.179, 0.211]",267057,0.05758,0.03832,0.21085,248.00000
"(0.211, 0.264]",267147,0.07208,0.04941,0.26972,232.00000


In [ ]:
group_time = train_df[["srch_id", "date_time"]].groupby("srch_id", sort=False)["date_time"].min().sort_values()
n_valid = max(1, int(len(group_time) * 0.1))
group_time.tail(n_valid).index.to_numpy()
set(group_time.tail(n_valid).index.to_numpy())


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score1,prop_location_score2,prop_log_historical_price,position,price_usd,promotion_flag,srch_destination_id,srch_length_of_stay,srch_booking_window,srch_adults_count,srch_children_count,srch_room_count,srch_saturday_night_bool,srch_query_affinity_score,orig_destination_distance,random_bool,comp1_rate,comp1_inv,comp1_rate_percent_diff,comp2_rate,comp2_inv,comp2_rate_percent_diff,comp3_rate,comp3_inv,comp3_rate_percent_diff,comp4_rate,comp4_inv,comp4_rate_percent_diff,comp5_rate,comp5_inv,comp5_rate_percent_diff,comp6_rate,comp6_inv,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool,relevance,search_month,search_dow,search_hour,party_size,rooms_per_person
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.50000,1,2.83000,0.04380,4.95000,27,104.77000,0,23246,1,0,4,0,1,1,NaN,NaN,1,<NA>,<NA>,NaN,0,0,NaN,0,0,NaN,<NA>,<NA>,NaN,0,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,0,NaN,0,NaN,0,0,4,3,8,4,0.25000
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.00000,1,2.20000,0.01490,5.03000,26,170.74001,0,23246,1,0,4,0,1,1,NaN,NaN,1,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,0,NaN,<NA>,<NA>,NaN,0,1,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,0,NaN,0,NaN,0,0,4,3,8,4,0.25000
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.50000,1,2.20000,0.02450,4.92000,21,179.80000,0,23246,1,0,4,0,1,1,NaN,NaN,1,<NA>,<NA>,NaN,0,0,NaN,0,0,NaN,<NA>,<NA>,NaN,0,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,0,NaN,0,NaN,0,0,4,3,8,4,0.25000
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.00000,1,2.83000,0.01250,4.39000,34,602.77002,0,23246,1,0,4,0,1,1,NaN,NaN,1,<NA>,<NA>,NaN,-1,0,5.00000,-1,0,5.00000,<NA>,<NA>,NaN,0,1,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,-1,0,5.00000,0,NaN,0,0,4,3,8,4,0.25000
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.50000,1,2.64000,0.12410,4.93000,4,143.58000,0,23246,1,0,4,0,1,1,NaN,NaN,1,<NA>,<NA>,NaN,0,0,NaN,0,0,NaN,<NA>,<NA>,NaN,0,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,0,NaN,0,NaN,0,0,4,3,8,4,0.25000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4958342,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,77700,3,4.00000,1,1.61000,0.04710,0.00000,2,118.00000,0,16974,1,21,3,0,1,0,NaN,550.91998,0,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,NaN,0,0,6,6,19,3,0.33333
4958343,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,88083,3,4.00000,1,1.95000,0.15200,0.00000,3,89.00000,0,16974,1,21,3,0,1,0,NaN,553.14001,0,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,NaN,0,0,6,6,19,3,0.33333
4958344,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,94508,3,3.50000,1,1.10000,0.01640,0.00000,4,99.00000,0,16974,1,21,3,0,1,0,NaN,544.42999,0,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,0,NaN,0,0,6,6,19,3,0.33333
4958345,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,128360,3,5.00000,1,1.95000,0.06620,0.00000,1,139.00000,0,16974,1,21,3,0,1,0,NaN,550.38000,0,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,1,157.84000,1,5,6,6,19,3,0.33333


## 10. Local Metric and Baseline Sanity Checks

Before modeling, verify the metric and baseline behavior. This uses complete `srch_id` groups and a time-aware validation split by search timestamp.


In [20]:
def make_relevance(df):
    return np.select([df["booking_bool"].eq(1), df["click_bool"].eq(1)], [5, 1], default=0).astype("int8")


def dcg_at_k(relevance, k=5):
    rel = np.asarray(relevance, dtype=float)[:k]
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
    return np.sum((np.power(2.0, rel) - 1.0) * discounts)


def ndcg_at_k(relevance, k=5):
    actual = dcg_at_k(relevance, k=k)
    ideal = dcg_at_k(sorted(relevance, reverse=True), k=k)
    return actual / ideal if ideal > 0 else 0.0


def mean_ndcg_at_k(df, score_col, k=5, group_col="srch_id", rel_col="relevance"):
    ordered = df.sort_values([group_col, score_col], ascending=[True, False])
    return ordered.groupby(group_col, observed=True)[rel_col].apply(lambda x: ndcg_at_k(x.to_numpy(), k=k)).mean()


def mean_ndcg_from_position(df, k=5):
    ordered = df.sort_values(["srch_id", "position"], ascending=[True, True])
    return ordered.groupby("srch_id", observed=True)["relevance"].apply(lambda x: ndcg_at_k(x.to_numpy(), k=k)).mean()

# Time-aware group split: searches are sorted by their first timestamp, then the latest groups are validation.
search_time = train_df.groupby("srch_id", observed=True)["date_time"].min().sort_values()
val_frac = 0.20
n_val_searches = int(len(search_time) * val_frac)
val_srch_ids = set(search_time.tail(n_val_searches).index)
metric_train = train_df.loc[~train_df["srch_id"].isin(val_srch_ids)].copy()
metric_val = train_df.loc[train_df["srch_id"].isin(val_srch_ids)].copy()

print(f"metric train rows/searches: {len(metric_train):,} / {metric_train['srch_id'].nunique():,}")
print(f"metric val rows/searches:   {len(metric_val):,} / {metric_val['srch_id'].nunique():,}")
print(f"train max date: {metric_train['date_time'].max()}")
print(f"val min date:   {metric_val['date_time'].min()}")


metric train rows/searches: 3,980,039 / 159,836
metric val rows/searches:   978,308 / 39,959
train max date: 2013-05-21 16:55:36
val min date:   2013-05-21 16:55:42


In [21]:
# Baseline 1: historical display position, diagnostic only because position is train-only.
position_ndcg = mean_ndcg_from_position(metric_val)
print(f"Diagnostic train display-position NDCG@5: {position_ndcg:.5f}")

# Baseline 2: random score for metric sanity.
rng = np.random.default_rng(RANDOM_STATE)
metric_val["random_score"] = rng.random(len(metric_val))
random_ndcg = mean_ndcg_at_k(metric_val, "random_score")
print(f"Random NDCG@5: {random_ndcg:.5f}")

# Baseline 3: property popularity learned only from metric_train.
global_relevance = metric_train["relevance"].mean()
prop_prior = (
    metric_train.groupby("prop_id", observed=True)
    .agg(rows=("relevance", "size"), relevance_mean=("relevance", "mean"), booking_rate=("booking_bool", "mean"), click_rate=("click_bool", "mean"))
    .reset_index()
)
alpha = 100
prop_prior["prop_pop_score"] = (prop_prior["relevance_mean"] * prop_prior["rows"] + global_relevance * alpha) / (prop_prior["rows"] + alpha)
metric_val = metric_val.merge(prop_prior[["prop_id", "prop_pop_score"]], on="prop_id", how="left")
metric_val["prop_pop_score"] = metric_val["prop_pop_score"].fillna(global_relevance)
prop_pop_ndcg = mean_ndcg_at_k(metric_val, "prop_pop_score")
print(f"Property popularity NDCG@5: {prop_pop_ndcg:.5f}")

# Baseline 4: simple within-search heuristic available in test.
heuristic_cols = ["prop_starrating", "prop_review_score", "prop_location_score1", "prop_location_score2", "promotion_flag"]
for c in heuristic_cols:
    metric_val[f"{c}_rank"] = metric_val.groupby("srch_id", observed=True)[c].rank(method="average", pct=True, ascending=False)
metric_val["price_rank_low"] = metric_val.groupby("srch_id", observed=True)["price_usd"].rank(method="average", pct=True, ascending=True)
metric_val["heuristic_score"] = (
    0.20 * (1 - metric_val["price_rank_low"].fillna(0.5))
    + 0.25 * metric_val["prop_starrating_rank"].fillna(0.5)
    + 0.25 * metric_val["prop_review_score_rank"].fillna(0.5)
    + 0.20 * metric_val["prop_location_score1_rank"].fillna(0.5)
    + 0.05 * metric_val["prop_location_score2_rank"].fillna(0.5)
    + 0.05 * metric_val["promotion_flag"].fillna(0)
)
heuristic_ndcg = mean_ndcg_at_k(metric_val, "heuristic_score")
print(f"Simple relative heuristic NDCG@5: {heuristic_ndcg:.5f}")

baseline_table = pd.DataFrame({
    "baseline": ["train display position (diagnostic/leaky)", "random", "property popularity", "simple relative heuristic"],
    "ndcg@5": [position_ndcg, random_ndcg, prop_pop_ndcg, heuristic_ndcg],
}).sort_values("ndcg@5", ascending=False)
display(baseline_table)


Diagnostic train display-position NDCG@5: 0.37496
Random NDCG@5: 0.15824
Property popularity NDCG@5: 0.27447
Simple relative heuristic NDCG@5: 0.14251


,baseline,ndcg@5
0,train display position (diagnostic/leaky),0.37496
2,property popularity,0.27447
1,random,0.15824
3,simple relative heuristic,0.14251


## 11. EDA Findings to Convert Into Features

MUST DO:

- Group validation by `srch_id`; never split rows from the same search across train and validation.
- Use relevance labels `5/1/0` and optimize local NDCG@5.
- Exclude `position`, `click_bool`, `booking_bool`, and `gross_bookings_usd` from final inference features.
- Add within-search ranks, percentiles, z-scores, and deltas for price, star rating, review score, location scores, promotion, and value-for-money.
- Add out-of-fold target encodings for `prop_id`, `srch_destination_id`, and `prop_id x srch_destination_id`.

HIGH IMPACT:

- Competitor missingness counts, cheaper/more-expensive counts, and percent-difference summaries.
- Temporal/search intent features from search month, day of week, hour, booking window, length of stay, party size, rooms, and weekend flag.
- Blend a LambdaRank model with a classifier/popularity score using within-search rank averaging.

RISK / LEAKAGE:

- Full-train target encodings are fine for final test inference only after validation design is fixed; they are invalid for validation scoring.
- `position` is useful for bias analysis but unavailable in test and should not be used in the final model.
- Tune against local validation first. Public leaderboard feedback is too noisy for feature selection by itself.


## 12. LLM-readable EDA artifacts

Save compact EDA outputs outside the notebook so they can be read by LLMs, scripts, and reviewers without executing the notebook. The main entry point is `artifacts/eda/eda_summary.json`, with CSV/JSON tables under `artifacts/eda/tables/`.


In [ ]:
import json
from datetime import datetime, timezone


def _repo_root_from_data_path(path):
    """Return repo root from data/processed/*.parquet paths."""
    resolved = Path(path).resolve()
    try:
        return resolved.parents[2]
    except IndexError:
        return Path.cwd().resolve()


EDA_ARTIFACT_DIR = _repo_root_from_data_path(TRAIN_PARQUET) / "artifacts" / "eda"
EDA_TABLE_DIR = EDA_ARTIFACT_DIR / "tables"
EDA_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EDA_TABLE_DIR.mkdir(parents=True, exist_ok=True)


def _clean_for_export(df):
    out = df.copy()
    if out.index.name is not None or not isinstance(out.index, pd.RangeIndex):
        out = out.reset_index()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].dt.strftime("%Y-%m-%dT%H:%M:%S")
        elif isinstance(out[col].dtype, pd.CategoricalDtype):
            out[col] = out[col].astype(str)
        elif out[col].dtype == "object":
            out[col] = out[col].map(lambda x: str(x) if not pd.isna(x) else None)
    out.columns = [str(c) for c in out.columns]
    return out


def _records(df, max_rows=200):
    if df is None:
        return []
    out = _clean_for_export(df).head(max_rows)
    return json.loads(out.to_json(orient="records", date_format="iso"))


def save_eda_table(name, df, max_rows=None):
    """Save a table as CSV and JSON records, returning metadata for the manifest."""
    out = _clean_for_export(df)
    if max_rows is not None:
        out = out.head(max_rows)
    csv_path = EDA_TABLE_DIR / f"{name}.csv"
    json_path = EDA_TABLE_DIR / f"{name}.json"
    out.to_csv(csv_path, index=False)
    out.to_json(json_path, orient="records", indent=2, date_format="iso")
    return {
        "name": name,
        "rows": int(len(out)),
        "columns": list(out.columns),
        "csv": str(csv_path.relative_to(EDA_ARTIFACT_DIR.parent.parent)),
        "json": str(json_path.relative_to(EDA_ARTIFACT_DIR.parent.parent)),
    }


saved_tables = []
for table_name, table_obj, max_rows in [
    ("schema_summary", globals().get("schema_summary"), None),
    ("train_only_usage", globals().get("train_only_usage"), None),
    ("dtype_missingness", globals().get("dtype_table"), None),
    ("label_summary", globals().get("label_summary"), None),
    ("query_summary", globals().get("query_summary"), None),
    ("query_label_patterns", globals().get("query_label_patterns"), None),
    ("candidate_count_distribution", globals().get("candidate_count_dist"), None),
    ("position_summary", globals().get("position_summary"), None),
    ("position_by_random", globals().get("position_by_random"), None),
    ("missingness_top", globals().get("miss"), None),
    ("row_missingness", globals().get("row_missing"), None),
    ("competitor_summary", globals().get("comp_summary"), None),
    ("id_coverage", globals().get("coverage"), None),
    ("cardinality", globals().get("cardinality"), None),
    ("numeric_summary", globals().get("num_summary"), None),
    ("numeric_drift", globals().get("drift"), None),
    ("price_quality", globals().get("price_quality"), None),
    ("top_property_popularity", globals().get("prop_pop"), 100),
    ("top_destination_popularity", globals().get("dest_pop"), 100),
    ("top_property_destination_popularity", globals().get("prop_dest_pop"), 100),
    ("baseline_ndcg", globals().get("baseline_table"), None),
]:
    if isinstance(table_obj, pd.DataFrame):
        saved_tables.append(save_eda_table(table_name, table_obj, max_rows=max_rows))

# Store target-rate segment tables that were displayed in loops earlier in the notebook.
segment_cols = [
    "search_month",
    "search_dow",
    "search_hour",
    "srch_length_of_stay",
    "srch_booking_window",
    "party_size",
    "srch_room_count",
    "srch_saturday_night_bool",
]
for col in segment_cols:
    if col in train_df.columns:
        target_by_segment = train_df.groupby(col, observed=True).agg(
            rows=("srch_id", "size"),
            click_rate=("click_bool", "mean"),
            booking_rate=("booking_bool", "mean"),
            relevance_mean=("relevance", "mean"),
        )
        target_by_segment["row_pct"] = target_by_segment["rows"] / len(train_df)
        saved_tables.append(save_eda_table(f"target_by_{col}", target_by_segment))

# Store compact search-relative diagnostics. These explain why relative ranking features were created.
if "rel" in globals() and "target_by_quantile" in globals():
    relative_diagnostic_cols = [
        "price_usd_rank_pct_asc",
        "price_usd_diff_median",
        "prop_starrating_rank_pct_desc",
        "prop_review_score_rank_pct_desc",
        "prop_location_score1_rank_pct_desc",
        "value_star_per_log_price",
        "value_review_per_log_price",
    ]
    for col in relative_diagnostic_cols:
        if col in rel.columns:
            saved_tables.append(
                save_eda_table(f"target_by_quantile_{col}", target_by_quantile(rel, col))
            )

key_findings = [
    "The task should be modeled as grouped ranking: srch_id is the query and prop_id is the hotel candidate.",
    "Bookings are much stronger feedback than clicks, so relevance is encoded as booking=5, click-only=1, none=0.",
    "Labels are sparse, which makes ranking metrics and strong validation more informative than row-wise accuracy.",
    "Train-only columns such as position, click_bool, booking_bool, and gross_bookings_usd must not be final prediction features.",
    "Position is useful for bias analysis but is not safe as a model feature because it reflects historical display order and is absent from test.",
    "Many visitor-history and competitor columns are highly missing, so missingness indicators and compact competitor summaries are useful features.",
    "Raw price and quality are context-dependent; within-search ranks, differences, and z-scores make features comparable inside each result list.",
    "High-cardinality IDs motivate frequency, affinity, and leakage-safe target encodings rather than large one-hot encodings.",
]

modeling_implications = [
    "Use group-based validation by srch_id to keep complete search result lists together.",
    "Use NDCG@5 because it rewards placing booked/clicked hotels near the top of each search.",
    "Fit target encodings out-of-fold for training rows and only from the training split for validation/test rows.",
    "Compare ranker results against random, popularity, and simple heuristic baselines before trusting model gains.",
]

base_schema = globals().get("schema_summary")
if isinstance(base_schema, pd.DataFrame) and {"train", "test"}.issubset(set(base_schema.index)):
    base_train_columns = int(base_schema.loc["train", "columns"])
    base_test_columns = int(base_schema.loc["test", "columns"])
else:
    # Fallback for running this cell alone after earlier feature-mutating cells.
    derived_cols = {"relevance", "search_month", "search_dow", "search_hour", "party_size", "rooms_per_person"}
    base_train_columns = int(len([c for c in train_df.columns if c not in derived_cols]))
    base_test_columns = int(len([c for c in test_df.columns if c not in derived_cols]))

summary = {
    "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "artifact_purpose": "LLM-readable EDA outputs for interpreting the Expedia hotel ranking project without parsing notebook outputs.",
    "data_paths": {
        "train_parquet": str(Path(TRAIN_PARQUET).resolve()),
        "test_parquet": str(Path(TEST_PARQUET).resolve()),
    },
    "dataset": {
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "train_columns_raw": base_train_columns,
        "test_columns_raw": base_test_columns,
        "train_searches": int(train_df["srch_id"].nunique()),
        "test_searches": int(test_df["srch_id"].nunique()),
        "train_properties": int(train_df["prop_id"].nunique()),
        "test_properties": int(test_df["prop_id"].nunique()),
        "avg_candidates_per_train_search": float(len(train_df) / train_df["srch_id"].nunique()),
    },
    "train_only_columns": list(train_only_cols),
    "labeling": {
        "relevance_definition": {"booking_bool=1": 5, "click_bool=1 and booking_bool=0": 1, "otherwise": 0},
        "click_rate": float(train_df["click_bool"].mean()),
        "booking_rate": float(train_df["booking_bool"].mean()),
        "label_summary": _records(label_summary),
        "query_summary": _records(query_summary),
    },
    "baseline_metrics": _records(globals().get("baseline_table")) if "baseline_table" in globals() else [],
    "key_findings": key_findings,
    "modeling_implications": modeling_implications,
    "tables": saved_tables,
}

summary_path = EDA_ARTIFACT_DIR / "eda_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

readme = f"""# EDA Artifacts\n\nGenerated from `notebooks/01_eda.ipynb` at `{summary['created_at_utc']}`.\n\nStart with `eda_summary.json`; it contains the compact narrative, core dataset facts, leakage notes, label definition, baseline metrics, and a manifest of saved tables. Detailed tables are saved as both CSV and JSON records under `tables/`.\n\n## Key Findings\n\n"""
readme += "\n".join(f"- {finding}" for finding in key_findings)
readme += "\n\n## Modeling Implications\n\n"
readme += "\n".join(f"- {item}" for item in modeling_implications)
readme += "\n\n## Saved Tables\n\n"
readme += "\n".join(f"- `{table['name']}`: {table['rows']} rows, {len(table['columns'])} columns" for table in saved_tables)
readme += "\n"
(EDA_ARTIFACT_DIR / "README.md").write_text(readme, encoding="utf-8")

print(f"Wrote LLM-readable EDA artifacts to: {EDA_ARTIFACT_DIR}")
print(f"Main summary: {summary_path}")
print(f"Saved tables: {len(saved_tables)}")
